# Appium Login Flow — Interactive Notebook

Run cells one at a time. Before starting:

1. Appium server running: `appium` (see README.md section 4)
2. Emulator/device connected: `adb devices` (see README.md section 2)
3. Locators filled in in `core/config.py` (use Appium Inspector, README.md section 7)

This notebook lets you run the login flow step by step, take screenshots
after each action, and dump the page source to find/verify locators —
much faster than editing-and-rerunning a full script each time.

Run Jupyter from the project root so relative paths resolve correctly:
`jupyter notebook notebooks/appium_login.ipynb`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from io import BytesIO
from PIL import Image

from core import config
from core.login_flow import create_driver, login, is_logged_in, get_login_error_text

def screenshot(drv):
    png = drv.get_screenshot_as_png()
    return Image.open(BytesIO(png))

## 1. Start the session

This installs/launches the app (per `config.CAPABILITIES`) and connects.

In [ ]:
driver, wait = create_driver()
screenshot(driver)

## 2. Inspect the page source

Use this to find resource-ids/xpaths if you haven't filled in `core/config.py`
yet, or to verify a locator exists on the current screen.

In [ ]:
print(driver.page_source)

## 3. Fill in and submit the login form

In [ ]:
USERNAME = "testuser"
PASSWORD = "correct-password"

login(driver, wait, USERNAME, PASSWORD)
screenshot(driver)

## 4. Check the result

In [ ]:
if is_logged_in(driver, wait, timeout=10):
    print("Login succeeded ✅")
else:
    error = get_login_error_text(driver)
    print("Login failed ❌", f"— error shown: {error}" if error else "— no error text found, check locators")

screenshot(driver)

## 5. Try an invalid login (optional)

Re-run cell 1 first if the app doesn't return you to the login screen
automatically (e.g. via a logout button).

In [ ]:
login(driver, wait, "testuser", "wrong-password")
screenshot(driver)

In [ ]:
print("Error text:", get_login_error_text(driver))

## 6. End the session

Always run this when done — only one Appium session can hold the device at
a time (this will block Appium Inspector or another test run otherwise).

In [ ]:
driver.quit()